In [ ]:
# Skipped: not running on Colab
pass

In [ ]:
## Standard libraries
import os
import math
import time
import numpy as np
import copy

## Imports for plotting
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib
matplotlib.rcParams['lines.linewidth'] = 2.0
plt.style.use('ggplot')

In [ ]:
# ===== Task 1: sampling from the three distributions =====
np.random.seed(0)

def sample_gaussian_mixture(N, sigma, centers):
    """
    Sample N points from an equally-weighted 2D Gaussian mixture
    with per-component isotropic covariance sigma^2 * I.

    Generative story:
      1. pick a component uniformly,
      2. sample a Gaussian around its center.
    """
    M = centers.shape[0]
    component_idx = np.random.randint(0, M, size=N)
    chosen_centers = centers[component_idx]       # (N, 2)
    noise = sigma * np.random.randn(N, 2)         # (N, 2)
    return chosen_centers + noise


def sample_spiral(N):
    """
    Sample N points from the spiral:
        theta ~ U[0, 4*pi],   r | theta ~ N(theta/2, var=0.25)
    then convert polar (r, theta) -> Cartesian (x, y).
    """
    theta = np.random.uniform(0, 4 * np.pi, size=N)
    r = np.random.normal(loc=theta / 2, scale=np.sqrt(0.25), size=N)
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    return np.stack([x, y], axis=1)


# Draw N samples from each of the three distributions
N = 10000
centers = np.array([[0, 0], [0, 2], [2, 0], [2, 2]], dtype=float)

samples_gmm_a  = sample_gaussian_mixture(N, sigma=0.5, centers=centers)
samples_gmm_b  = sample_gaussian_mixture(N, sigma=1.0, centers=centers)
samples_spiral = sample_spiral(N)

# Display the drawn samples for each distribution separately
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, samples, title in zip(
    axes,
    [samples_gmm_a, samples_gmm_b, samples_spiral],
    ['Gaussian Mixture (sigma=0.5)', 'Gaussian Mixture (sigma=1)', 'Spiral'],
):
    ax.scatter(samples[:, 0], samples[:, 1], s=2, alpha=0.3)
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.set_xlabel(r'$x_1$'); ax.set_ylabel(r'$x_2$')
plt.tight_layout()
plt.show()


In [ ]:
# ===== Task 2: multivariate KDE with a 2D separable Gaussian kernel =====
def kde_estimate(query_points, samples, H):
    """
    Multivariate kernel density estimate at given query points:

        f_hat(x) = (1 / (N * |H|)) * sum_i K( H^{-1} (x_i - x) )

    with the 2D separable Gaussian kernel
        K(u) = k(u_1) * k(u_2),    k(u) = (1/sqrt(2*pi)) * exp(-u^2 / 2)
    i.e. K(u) = (1/(2*pi)) * exp(-||u||^2 / 2).

    Args:
        query_points (M, 2): points at which to evaluate the density.
        samples      (N, 2): i.i.d. samples drawn from f.
        H            (2, 2): bandwidth matrix.

    Returns:
        densities (M,): estimated density values at the query points.
    """
    H_inv = np.linalg.inv(H)
    det_H = np.abs(np.linalg.det(H))
    N = samples.shape[0]

    # diff[m, i] = x_i - x_m, shape (M, N, 2)
    diff = samples[None, :, :] - query_points[:, None, :]
    # apply H^{-1} to every 2-vector: (M, N, 2)
    u = diff @ H_inv.T
    # Gaussian kernel value at each (query, sample) pair: (M, N)
    K_vals = np.exp(-0.5 * (u ** 2).sum(axis=-1)) / (2 * np.pi)
    # average over samples, divide by |H|
    return K_vals.sum(axis=1) / (N * det_H)


In [ ]:
# ===== Task 3: compare f(x) and f_hat(x) for h in {0.1, 0.5, 1} =====

def gmm_pdf(X, Y, sigma, centers):
    """Analytic PDF of an equally-weighted isotropic 2D Gaussian mixture."""
    p = np.zeros_like(X, dtype=float)
    for mu in centers:
        sq = (X - mu[0])**2 + (Y - mu[1])**2
        p += np.exp(-sq / (2 * sigma**2)) / (2 * np.pi * sigma**2)
    return p / centers.shape[0]


def spiral_pdf(X, Y):
    """Analytic PDF of the spiral via change of variables (polar -> Cartesian)."""
    rho = np.sqrt(X**2 + Y**2)
    phi = np.arctan2(Y, X) % (2 * np.pi)              # in [0, 2*pi)
    sigma_r = np.sqrt(0.25)                            # std = 0.5

    p = np.zeros_like(rho, dtype=float)
    # positive-r pre-images: r = rho, theta in {phi, phi + 2*pi}
    for theta in [phi, phi + 2 * np.pi]:
        gauss = np.exp(-0.5 * ((rho - theta / 2) / sigma_r) ** 2) / (sigma_r * np.sqrt(2 * np.pi))
        p += gauss
    # negative-r pre-images: r = -rho, theta in {(phi+pi) mod 2*pi, that + 2*pi}
    phi_neg = (phi + np.pi) % (2 * np.pi)
    for theta in [phi_neg, phi_neg + 2 * np.pi]:
        gauss = np.exp(-0.5 * ((-rho - theta / 2) / sigma_r) ** 2) / (sigma_r * np.sqrt(2 * np.pi))
        p += gauss

    rho_safe = np.where(rho < 1e-6, 1e-6, rho)
    return p / (4 * np.pi * rho_safe)


def compare_kde(samples, true_pdf_fn, x_range, y_range,
                h_values=(0.1, 0.5, 1.0), title='', grid=100):
    """Plot true f(x) and f_hat(x) for each h side by side."""
    xs = np.linspace(x_range[0], x_range[1], grid)
    ys = np.linspace(y_range[0], y_range[1], grid)
    XX, YY = np.meshgrid(xs, ys)
    pts = np.stack([XX.ravel(), YY.ravel()], axis=1)
    extent = [x_range[0], x_range[1], y_range[0], y_range[1]]

    f_true = true_pdf_fn(XX, YY)

    fig, axes = plt.subplots(1, len(h_values) + 1, figsize=(4 * (len(h_values) + 1), 4))
    axes[0].imshow(f_true, extent=extent, origin='lower', aspect='auto', cmap='viridis')
    axes[0].set_title(f'{title}: $f(x)$')

    for ax, h in zip(axes[1:], h_values):
        H = np.array([[h, 0.0], [0.0, h]])
        f_hat = kde_estimate(pts, samples, H).reshape(grid, grid)
        ax.imshow(f_hat, extent=extent, origin='lower', aspect='auto', cmap='viridis')
        ax.set_title(rf'$\hat f(x)$, $h={h}$')

    plt.tight_layout()
    plt.show()


# GMM, sigma = 0.5
compare_kde(samples_gmm_a,
            lambda X, Y: gmm_pdf(X, Y, sigma=0.5, centers=centers),
            x_range=(-2, 4), y_range=(-2, 4),
            title=r'GMM ($\sigma=0.5$)')

# GMM, sigma = 1
compare_kde(samples_gmm_b,
            lambda X, Y: gmm_pdf(X, Y, sigma=1.0, centers=centers),
            x_range=(-3, 5), y_range=(-3, 5),
            title=r'GMM ($\sigma=1$)')

# Spiral
compare_kde(samples_spiral, spiral_pdf,
            x_range=(-7, 7), y_range=(-7, 7),
            title='Spiral')


In [ ]:
## Scikit-learn built-in dataset generators
from sklearn.datasets import make_moons, make_circles, make_blobs

## Progress bar
import tqdm

## PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import Adam
from torch.distributions.multivariate_normal import MultivariateNormal

# Traning will be done on CPU for this homework. 
# For K=4, N=1500, epochs=1000 takes < 3 mins.
device = torch.device("cpu")
print("Using device", device)

In [ ]:
def sample_2d_datasets(dist_type, num_samples=1000, seed=0, num_gaussians=5):
  """
  function samples from simple pre-defined distributions in 2D.
  Inputs:
    - dist_type: str specifying the distribution to be chosen from:
      {'Circles', 'Moons', 'GaussiansGrid', 'GaussiansRot'}
    - num_samples: Number of samples to draw from dist_type (int).
    - seed: Random seed integer.
    - num_gaussians: Number of rotated gaussians if dist_type='GaussiansRot'. 
      (relevant only for dist_type='GaussiansRot', should be a keyword argument)
  Outputs:
    - data (np.array): array of num_samplesx2 samples from dist_type
  """
  np.random.seed(seed)
  if dist_type == 'Circles':
    data = make_circles(num_samples, noise=.1, factor=.8, random_state=seed, shuffle=True)[0]
  elif dist_type == 'Moons':
    data = make_moons(num_samples, noise=.1, random_state=seed, shuffle=True)[0]
  elif dist_type == 'GaussiansGrid':
    centers = np.array([[0,0],[0,2],[2,0],[2,2]])
    data = make_blobs(num_samples, centers=centers, cluster_std=.5, random_state=seed, shuffle=True)[0]
  elif dist_type == 'GaussiansRot':
    angles = np.linspace(0, 2 * np.pi, num_gaussians, endpoint=False)
    centers = np.stack([2.5 * np.array([np.cos(angle), np.sin(angle)]) for angle in angles])
    data = make_blobs(num_samples, centers=centers, cluster_std=np.sqrt(.1), random_state=seed, shuffle=True)[0]
  else:
    raise NotImplementedError
  return data

In [ ]:
# ===== Part II, Task 1: sample N=1000 from each of the 4 toy datasets =====
dist_types = ['Circles', 'Moons', 'GaussiansGrid', 'GaussiansRot']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, dist_type in zip(axes, dist_types):
    data_pts = sample_2d_datasets(dist_type, num_samples=1000)
    ax.scatter(data_pts[:, 0], data_pts[:, 1], s=4, alpha=0.5)
    ax.set_title(dist_type)
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()


In [ ]:
class ToyDataset(Dataset):
  def __init__(self, dist_type, num_samples=1000, seed=0, num_gaussians=5):
      """
      Wrapper around the function "sample_2d_datasets" to allow iterating 
      batches using a datalaoder when training our normalizing flow model.
      """
      self.data = sample_2d_datasets(dist_type, num_samples, seed, num_gaussians)
      self.num_samples = num_samples

  def __len__(self):
      return self.num_samples

  def __getitem__(self, index):
      return torch.from_numpy(self.data[index]).type(torch.FloatTensor)

In [ ]:
class CouplingLayer(nn.Module):
  def __init__(self, mask):
    super(CouplingLayer, self).__init__()

    # mask for splitting (fixed not learnable)
    self.mask = nn.Parameter(mask, requires_grad=False)

    # scaling function and stabilizing scale_factor init. to 0
    self.s_func = nn.Sequential(nn.Linear(in_features=2, out_features=32), 
                                nn.LeakyReLU(),
                                nn.Linear(in_features=32, out_features=32), 
                                nn.LeakyReLU(),
                                nn.Linear(in_features=32, out_features=2))
    self.scale_factor = nn.Parameter(torch.Tensor(2).fill_(0.0))

    # shifting function
    self.t_func = nn.Sequential(nn.Linear(in_features=2, out_features=32), 
                                nn.LeakyReLU(),
                                nn.Linear(in_features=32, out_features=32), 
                                nn.LeakyReLU(),
                                nn.Linear(in_features=32, out_features=2))

  def forward(self, x):
    # use x_{1:j} (the mask==1 part) to predict s and t
    x_masked = x * self.mask
    s = self.s_func(x_masked) * self.scale_factor
    t = self.t_func(x_masked)
    # apply s, t only to the transformed part (mask==0)
    s = s * (1 - self.mask)
    t = t * (1 - self.mask)
    y = x * torch.exp(s) + t
    log_det_jac = s.sum(dim=-1)
    return y, log_det_jac

  def inverse(self, y):
    # y_{1:j} == x_{1:j}, so we recompute s and t from y * mask
    y_masked = y * self.mask
    s = self.s_func(y_masked) * self.scale_factor
    t = self.t_func(y_masked)
    s = s * (1 - self.mask)
    t = t * (1 - self.mask)
    x = (y - t) * torch.exp(-s)
    inv_log_det_jac = -s.sum(dim=-1)
    return x, inv_log_det_jac


In [ ]:
class CouplingFlow(nn.Module):
  def __init__(self, num_layers):
      super(CouplingFlow, self).__init__()

      # concatenate coupling layers with alternating masks
      masks = F.one_hot(torch.tensor([i % 2 for i in range(num_layers)])).float()
      self.layers = nn.ModuleList([CouplingLayer(mask) for mask in masks])

      # define prior distribution to be z~N(0,I)
      self.prior = MultivariateNormal(torch.zeros(2),torch.eye(2))

  def log_probability(self, x):
    # x = f_K o ... o f_1 (z), so to get log p_X(x):
    #   apply f_K^{-1}, ..., f_1^{-1} to x and accumulate inv-log-det-jacs.
    sum_inv_log_det = torch.zeros(x.shape[0])
    z = x
    for layer in reversed(self.layers):
      z, inv_log_det = layer.inverse(z)
      sum_inv_log_det = sum_inv_log_det + inv_log_det
    log_prob = self.prior.log_prob(z) + sum_inv_log_det
    return log_prob

  def sample_x(self, num_samples):
    z = self.prior.sample((num_samples,))
    log_prob = self.prior.log_prob(z)
    x = z
    for layer in self.layers:
      x, log_det = layer.forward(x)
      # log p_X(x) = log p_Z(z) - sum_k log|det df_k/d(.)|
      log_prob = log_prob - log_det
    return x, log_prob

  def sample_x_each_step(self, num_samples):
    z = self.prior.sample((num_samples,))
    samples = [z]
    x = z
    for layer in self.layers:
      x, _ = layer.forward(x)
      samples.append(x)
    return samples


In [ ]:
# detach tensor and transfer to numpy
def to_np(x):
  return x.detach().numpy()

# Simple training function
def train(model, data, epochs = 100, batch_size = 64):

  # move model into the device
  model = model.to(device)

  # split into training and validation, and create the loaders
  lengths = [int(len(data)*0.9), len(data) - int(len(data)*0.9)]
  train_set, valid_set = random_split(data, lengths)
  train_loader = DataLoader(train_set, batch_size=batch_size)
  valid_loader = DataLoader(valid_set, batch_size=batch_size)

  # define the optimizer and scheduler
  optimizer = Adam(model.parameters(), lr=1e-3)

  # train the model
  train_losses, valid_losses, min_valid_loss = [], [], np.inf
  with tqdm.tqdm(range(epochs), unit=' Epoch') as tepoch:
    for epoch in tepoch:

      # training loop
      epoch_loss = 0
      model.train(True)
      for batch_index, training_sample in enumerate(train_loader):
          log_prob = model.log_probability(training_sample)
          loss = - log_prob.mean(0)
          optimizer.zero_grad()
          loss.backward()
          optimizer.step()
          epoch_loss += loss
      epoch_loss /= len(train_loader)
      train_losses.append(np.copy(to_np(epoch_loss)))

      # validation loop
      epoch_loss_valid = 0
      model.train(False)
      for batch_index, valid_sample in enumerate(valid_loader):
        log_prob = model.log_probability(valid_sample)
        loss_valid = - log_prob.mean(0)
        epoch_loss_valid += loss_valid
      
      epoch_loss_valid /= len(valid_loader)
      valid_losses.append(np.copy(to_np(epoch_loss_valid)))
      
      # save best model based off validation loss
      if epoch_loss_valid < min_valid_loss:
        model_best = copy.deepcopy(model)
        min_valid_loss = epoch_loss_valid
        epoch_min = epoch

      # report progress with tqdm pbar
      tepoch.set_postfix(train_loss=to_np(epoch_loss), valid_loss=to_np(epoch_loss_valid))
      
  # report best model on val.
  print('\n Best Model achieved {:.4f} validation loss at epoch {} \n'.
        format(min_valid_loss, epoch_min))

  # if the number of samples is too low take the final weights regardless of 
  # valdiation loss due to weak statistics (overfitting avoided by early stopping)
  if lengths[1] < 500:
    model_best = model

  return model_best, train_losses, valid_losses

In [ ]:
# seeds to ensure reproducibility
torch.manual_seed(8)
np.random.seed(0)

# dataset
num_samples = 1500
data = ToyDataset('Moons', num_samples=num_samples)

# learning hyper-parameters
K = 4
nepochs = 1000

# instantiate model and optimize the parameters
Flow_model = CouplingFlow(num_layers=K)
moon_model, train_loss, valid_loss = train(Flow_model, data, epochs=nepochs)

In [ ]:
# ===== Task 9: train K=4 flow on each of the 4 datasets, plot density and intermediate samples =====

def _data_bounds(data, pad=0.5):
    pts = data.data
    return (pts[:, 0].min() - pad, pts[:, 0].max() + pad,
            pts[:, 1].min() - pad, pts[:, 1].max() + pad)


def plot_flow_density(model, data, title='', grid_size=100):
    x_min, x_max, y_min, y_max = _data_bounds(data)
    xs = np.linspace(x_min, x_max, grid_size)
    ys = np.linspace(y_min, y_max, grid_size)
    XX, YY = np.meshgrid(xs, ys)
    grid = np.stack([XX.ravel(), YY.ravel()], axis=1).astype(np.float32)

    model.eval()
    with torch.no_grad():
        log_prob = model.log_probability(torch.from_numpy(grid))
    p = torch.exp(log_prob).numpy().reshape(grid_size, grid_size)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(p, extent=[x_min, x_max, y_min, y_max],
              origin='lower', aspect='auto', cmap='viridis')
    ax.set_title(title)
    plt.show()


def plot_flow_intermediate(model, data, num_samples=1000, title=''):
    model.eval()
    with torch.no_grad():
        samples_list = model.sample_x_each_step(num_samples)

    x_min, x_max, y_min, y_max = _data_bounds(data)
    K = len(samples_list)
    fig, axes = plt.subplots(1, K, figsize=(3 * K, 3.5), sharex=True, sharey=True)
    for k, (ax, s) in enumerate(zip(axes, samples_list)):
        s_np = s.numpy()
        ax.scatter(s_np[:, 0], s_np[:, 1], s=4, alpha=0.5)
        if k == 0:
            t = r'$p(z)$'
        elif k == K - 1:
            t = r'$p(x)$'
        else:
            t = f'after layer {k}'
        ax.set_title(t)
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
    if title:
        fig.suptitle(title, y=1.05)
    plt.tight_layout()
    plt.show()


dist_configs = [
    ('Circles',       500),
    ('Moons',        1000),
    ('GaussiansGrid', 500),
    ('GaussiansRot',  500),
]
N = 1500
K = 4

trained_models = {}
for dist_type, nepochs in dist_configs:
    print(f'\n=== {dist_type} ({nepochs} epochs) ===')
    torch.manual_seed(8)
    np.random.seed(0)
    data = ToyDataset(dist_type, num_samples=N)
    flow = CouplingFlow(num_layers=K)
    best, _, _ = train(flow, data, epochs=nepochs)
    trained_models[dist_type] = (best, data)
    plot_flow_density(best, data, title=f'{dist_type}: estimated $p(x)$')
    plot_flow_intermediate(best, data, num_samples=1000, title=dist_type)


In [ ]:
# ===== Task 10: effect of model depth K in {2, 4} on GaussiansRot =====
N_t10 = 1500
nepochs_t10 = 250

models_by_K = {}
for K_t10 in (2, 4):
    print(f'\n=== K = {K_t10} ===')
    torch.manual_seed(8)
    np.random.seed(0)
    data = ToyDataset('GaussiansRot', num_samples=N_t10)
    flow = CouplingFlow(num_layers=K_t10)
    best, _, _ = train(flow, data, epochs=nepochs_t10)
    models_by_K[K_t10] = (best, data)
    plot_flow_density(best, data, title=f'GaussiansRot, $K={K_t10}$: $p(x)$')
    plot_flow_intermediate(best, data, num_samples=1000, title=f'$K={K_t10}$')


In [ ]:
# ===== Task 11: effect of training set size N in {1500, 3000} on GaussiansRot =====
K_t11 = 4
nepochs_t11 = 400

models_by_N = {}
for N_t11 in (1500, 3000):
    print(f'\n=== N = {N_t11} ===')
    torch.manual_seed(8)
    np.random.seed(0)
    data = ToyDataset('GaussiansRot', num_samples=N_t11)
    flow = CouplingFlow(num_layers=K_t11)
    best, _, _ = train(flow, data, epochs=nepochs_t11)
    models_by_N[N_t11] = (best, data)
    plot_flow_density(best, data, title=f'GaussiansRot, $N={N_t11}$: $p(x)$')
    plot_flow_intermediate(best, data, num_samples=1000, title=f'$N={N_t11}$')


In [ ]:
# ===== Task 12: effect of data complexity (num_gaussians in {3, 7}) =====
N_t12 = 1500
K_t12 = 4
nepochs_t12 = 250

models_by_ng = {}
for ng in (3, 7):
    print(f'\n=== num_gaussians = {ng} ===')
    torch.manual_seed(8)
    np.random.seed(0)
    data = ToyDataset('GaussiansRot', num_samples=N_t12, num_gaussians=ng)
    flow = CouplingFlow(num_layers=K_t12)
    best, _, _ = train(flow, data, epochs=nepochs_t12)
    models_by_ng[ng] = (best, data)
    plot_flow_density(best, data, title=f'GaussiansRot, $M={ng}$: $p(x)$')
    plot_flow_intermediate(best, data, num_samples=1000, title=f'$M={ng}$')
